# Tarea 1 – Parte 2: Modelado y clasificación de titulares

- **Estudiante:** *(tu nombre y @usuario_github)*
- **Medio:** El Comercio · **Categorías:** `politica`, `economia`, `deportes`, `ciencia_tecnologia`
- **Entorno:** Google Colab con GPU T4 (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*)

**Qué hace este cuaderno, en orden**

1. Carga `datos/noticias.csv` y hace la **división estratificada 70/15/15** (`random_state=42`).
2. **Zero-shot** sobre 40 titulares del conjunto de prueba (sin entrenar nada).
3. **Etiquetado débil con un LLM** (Gemini) + **auditoría humana de 20 casos** → `datos/etiquetas_llm.csv`.
4. **Fine-tuning** de `distilbert-base-multilingual-cased` con `Trainer` (3 épocas).
5. **Evaluación en test**: accuracy, F1 macro, matriz de confusión y reflexión por categoría.
6. **Tabla comparativa** y justificación.

**Antes de ejecutar:** en Colab, ícono de la llave (Secretos) → agrega `GEMINI_API_KEY` y activa «Acceso desde el cuaderno». Opcional: `HF_TOKEN` si quieres subir el modelo a Hugging Face.

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn seaborn google-genai huggingface_hub

In [ ]:
import json, os, subprocess, time, warnings
from getpass import getpass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42

# Si estamos en Colab, descargamos el repositorio (los datos ya están subidos a GitHub).
if not os.path.exists("datos/noticias.csv"):
    if not os.path.exists("Cursos"):
        subprocess.run(["git", "clone", "https://github.com/gabriela-pozo/Cursos.git"], check=True)
    os.chdir("Cursos/tarea_1")
os.makedirs("modelo", exist_ok=True)

def leer_secreto(nombre):
    """Busca la clave en los Secretos de Colab, luego en variables de entorno; si no, la pide."""
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return os.getenv(nombre) or getpass(f"{nombre}: ")

print("GPU disponible:", torch.cuda.is_available(), "| carpeta:", os.getcwd())

## 1. Datos y división train / validación / test

**Teoría.** Un modelo se evalúa con datos que **nunca vio**. Por eso dividimos en tres:

| Conjunto | Para qué sirve | Tamaño |
|---|---|---|
| **train** | el modelo aprende de aquí | 70 % |
| **validación** | elegir época/hiperparámetros durante el entrenamiento | 15 % |
| **test** | nota final, se usa **una sola vez** al final | 15 % |

- **Estratificada:** cada conjunto mantiene la misma proporción de categorías que el total (si no, el test podría quedar sin deportes, por ejemplo).
- **`random_state=42`:** fija el azar para que cualquiera que repita el cuaderno obtenga la misma división.
- **Fuga de datos (*leakage*):** si un titular casi idéntico está en train y en test, el modelo «hace trampa». Por eso en la Parte 1 se eliminaron los duplicados; aquí lo comprobamos otra vez.

In [ ]:
ETIQUETAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
ID2LABEL = dict(enumerate(ETIQUETAS))
LABEL2ID = {e: i for i, e in ID2LABEL.items()}

df = pd.read_csv("datos/noticias.csv").fillna({"bajada": ""})
assert set(df["etiqueta"]) == set(ETIQUETAS), "hay etiquetas fuera de las 4 categorías"
df["y"] = df["etiqueta"].map(LABEL2ID)
print("Total:", df.shape)
df["etiqueta"].value_counts()

In [ ]:
train, temp = train_test_split(df, test_size=0.30, stratify=df["etiqueta"], random_state=SEED)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["etiqueta"], random_state=SEED)
train, val, test = (x.reset_index(drop=True) for x in (train, val, test))

# Comprobaciones: sin solapes por URL ni por titular normalizado.
norm = lambda s: s.str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
for a, b, nombre in [(train, val, "train-val"), (train, test, "train-test"), (val, test, "val-test")]:
    assert not set(a["url"]) & set(b["url"]), f"URL repetida entre {nombre}"
    assert not set(norm(a["titular"])) & set(norm(b["titular"])), f"titular repetido entre {nombre}"

division = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")], ignore_index=True)
division[["url", "split"]].to_csv("datos/split.csv", index=False)
pd.crosstab(division["etiqueta"], division["split"], margins=True)

## 2. Métricas: accuracy y F1 macro

- **Accuracy** = aciertos / total. Es fácil de leer, pero engaña si las clases están desbalanceadas.
- **Precisión** (de lo que dije «deportes», ¿cuánto lo era?) y **recall** (de todo lo que era deportes, ¿cuánto encontré?).
- **F1** = media armónica de precisión y recall para **una** clase. **F1 macro** = promedio simple de los F1 de las 4 clases: todas pesan igual, y por eso es la métrica oficial de la tarea.

In [ ]:
def metricas(y_real, y_pred):
    return {"accuracy": accuracy_score(y_real, y_pred),
            "f1_macro": f1_score(y_real, y_pred, average="macro")}

## 3. Zero-shot (sin entrenar)

**Teoría.** Un modelo *zero-shot* clasifica en categorías que nunca vio etiquetadas. Se apoya en un modelo de **inferencia de lenguaje natural (NLI)**: para cada categoría arma una hipótesis («Este titular trata sobre *deportes*») y pregunta si el titular la implica. La categoría con mayor probabilidad gana.

Modelo: `joeddav/xlm-roberta-large-xnli` (multilingüe, incluye español). Lo probamos sobre **40 titulares del conjunto de prueba** (10 por categoría, semilla 42).

In [ ]:
from transformers import pipeline

NOMBRES_ES = {"politica": "política", "economia": "economía", "deportes": "deportes",
              "ciencia_tecnologia": "ciencia y tecnología"}
ES2ETIQ = {v: k for k, v in NOMBRES_ES.items()}

muestra40 = pd.concat([g.sample(min(len(g), 10), random_state=SEED) for _, g in test.groupby("etiqueta")]).reset_index(drop=True)
print("Titulares en la muestra:", len(muestra40))

t0 = time.time()
zs = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli",
              device=0 if torch.cuda.is_available() else -1)
salidas = zs(list(muestra40["titular"]), candidate_labels=list(NOMBRES_ES.values()),
             hypothesis_template="Este titular trata sobre {}.", batch_size=8)
muestra40["pred_zs"] = [ES2ETIQ[s["labels"][0]] for s in salidas]
tiempo_zs = time.time() - t0

res_zs = metricas(muestra40["etiqueta"], muestra40["pred_zs"])
print(f"Zero-shot (40 titulares) -> accuracy {res_zs['accuracy']:.3f} | F1 macro {res_zs['f1_macro']:.3f} | {tiempo_zs:.0f} s")
muestra40.loc[muestra40["etiqueta"] != muestra40["pred_zs"], ["titular", "etiqueta", "pred_zs"]]

## 4. Etiquetado débil con un LLM y auditoría humana

**Teoría.** En la práctica etiquetar a mano miles de textos es caro. Un LLM puede dar etiquetas **rápidas y baratas pero imperfectas**: se llaman *etiquetas débiles* (*weak labels*). Por eso nunca se confía a ciegas: se **audita una muestra** a mano y se mide el acuerdo.

Aquí el LLM (Gemini) etiqueta **los 385 titulares** usando solo el titular, en lotes de 20 para ahorrar llamadas. Las etiquetas de entrenamiento del modelo siguen siendo las de la Parte 1 (sección del medio); el LLM sirve para **contrastarlas** y para la auditoría. Guardamos `datos/etiquetas_llm.csv`.

> La clave se lee de los Secretos de Colab (`GEMINI_API_KEY`). **Nunca la escribas en el cuaderno.** Si el nombre del modelo ya no existe, revisa los vigentes en https://ai.google.dev/gemini-api/docs/models y cambia `MODELO_LLM`.

In [ ]:
from google import genai

MODELO_LLM = "gemini-2.5-flash"
RUTA_LLM = "datos/etiquetas_llm.csv"

PROMPT = """Eres un editor de un diario peruano. Clasifica cada titular en UNA sola categoría:
- politica: gobierno, Congreso, elecciones, partidos, justicia y temas políticos
- economia: dinero, precios, empresas, empleo, bolsa, dólar, impuestos
- deportes: fútbol, otros deportes, competiciones, deportistas
- ciencia_tecnologia: ciencia, tecnología, internet, celulares, IA, espacio, salud-ciencia

Responde SOLO con un JSON válido: una lista de objetos {"id": <número>, "etiqueta": "<categoría>", "confianza": <0 a 1>}.

Titulares:
"""

def etiquetar_lote(cliente, titulares, intentos=3):
    texto = PROMPT + "\n".join(f"{i}. {t}" for i, t in enumerate(titulares))
    for n in range(intentos):
        try:
            r = cliente.models.generate_content(model=MODELO_LLM, contents=texto,
                                                config={"response_mime_type": "application/json", "temperature": 0})
            datos = {d["id"]: d for d in json.loads(r.text)}
            return [(datos[i]["etiqueta"], float(datos[i]["confianza"])) for i in range(len(titulares))]
        except Exception as e:
            print(f"  intento {n + 1} falló: {str(e)[:120]}")
            time.sleep(10 * (n + 1))
    return [("sin_respuesta", 0.0)] * len(titulares)

if os.path.exists(RUTA_LLM):
    llm = pd.read_csv(RUTA_LLM)
    print("Se reutiliza", RUTA_LLM, llm.shape)
else:
    cliente = genai.Client(api_key=leer_secreto("GEMINI_API_KEY"))
    filas = []
    titulares = df["titular"].tolist()
    for ini in range(0, len(titulares), 20):
        lote = titulares[ini:ini + 20]
        for t, (et, conf) in zip(lote, etiquetar_lote(cliente, lote)):
            filas.append({"titular": t, "etiqueta_llm": et, "confianza": conf,
                          "modelo_llm": MODELO_LLM, "fecha": time.strftime("%Y-%m-%d")})
        print(f"  {min(ini + 20, len(titulares))}/{len(titulares)}")
        time.sleep(4)
    llm = pd.DataFrame(filas)
    llm.to_csv(RUTA_LLM, index=False)

print("Etiquetas inválidas o sin respuesta:", (~llm["etiqueta_llm"].isin(ETIQUETAS)).sum())
llm["etiqueta_llm"].value_counts()

In [ ]:
# ¿Cuánto coincide el LLM con las etiquetas de la Parte 1 (sección del medio)?
comp = df[["titular", "etiqueta"]].merge(llm, on="titular", how="left")
acuerdo_total = (comp["etiqueta"] == comp["etiqueta_llm"]).mean()
print(f"Acuerdo LLM vs sección del medio (385 titulares): {acuerdo_total:.1%}")

en_test = comp[comp["titular"].isin(test["titular"])]
res_llm_test = metricas(en_test["etiqueta"], en_test["etiqueta_llm"].where(en_test["etiqueta_llm"].isin(ETIQUETAS), "politica"))
print(f"LLM sobre el test -> accuracy {res_llm_test['accuracy']:.3f} | F1 macro {res_llm_test['f1_macro']:.3f}")

# Desacuerdos: ¿quién tiene razón, el LLM o la sección?
comp.loc[comp["etiqueta"] != comp["etiqueta_llm"], ["titular", "etiqueta", "etiqueta_llm", "confianza"]].head(15)

### Auditoría humana de 20 titulares al azar

Haz la auditoría **a ciegas** (sin ver la etiqueta del LLM) para no sesgarte:

1. Ejecuta la celda siguiente: te muestra 20 titulares numerados.
2. Léelos y escribe **tú** la categoría de cada uno en la lista `MIS_ETIQUETAS` (de la celda que sigue), en el mismo orden. Valores permitidos: `politica`, `economia`, `deportes`, `ciencia_tecnologia`.
3. Ejecuta la última celda: calcula el acuerdo `xx/20` y te muestra los casos donde difieres.

In [ ]:
auditoria = llm.sample(20, random_state=SEED).reset_index(drop=True)
for i, t in enumerate(auditoria["titular"]):
    print(f"{i:>2}. {t}")

In [ ]:
# Rellena con TU criterio, en el orden mostrado arriba (20 valores).
MIS_ETIQUETAS = [
    "", "", "", "", "",
    "", "", "", "", "",
    "", "", "", "", "",
    "", "", "", "", "",
]

validas = [e in ETIQUETAS for e in MIS_ETIQUETAS]
if len(MIS_ETIQUETAS) != 20 or not all(validas):
    print("Aún faltan etiquetas o hay valores no permitidos. Revisa la lista.")
    acuerdo_20 = None
else:
    auditoria["etiqueta_humana"] = MIS_ETIQUETAS
    acuerdo_20 = int((auditoria["etiqueta_humana"] == auditoria["etiqueta_llm"]).sum())
    print(f"Acuerdo LLM-Humano: {acuerdo_20}/20")
    auditoria.loc[auditoria["etiqueta_humana"] != auditoria["etiqueta_llm"], ["titular", "etiqueta_humana", "etiqueta_llm", "confianza"]]

**Tu análisis (escríbelo aquí, 3–4 líneas):** ¿En qué casos se equivocó el LLM? ¿Eran titulares ambiguos (por ejemplo, economía del deporte o política tecnológica)? ¿Tenía menos confianza en esos? ¿Confiarías en este etiquetado sin auditoría para entrenar un modelo?

## 5. Fine-tuning de DistilBERT multilingüe

**Teoría.**
- **Transformer preentrenado:** `distilbert-base-multilingual-cased` ya «sabe español» porque leyó millones de textos. No lo entrenamos desde cero.
- **Fine-tuning:** le pegamos una capa final de clasificación (4 salidas) y seguimos entrenando con **nuestros** titulares para que ajuste sus pesos a la tarea.
- **Tokenización:** el texto se parte en *tokens* (subpalabras) y se convierte en números; `max_length=64` basta para un titular.
- **Época:** una pasada completa por el train. Usamos **3**. Pocas épocas: no aprende; demasiadas: **sobreajusta** (memoriza el train y falla en datos nuevos). Por eso miramos la **validación** al final de cada época y nos quedamos con la mejor.
- **Batch size 16:** cuántos ejemplos procesa antes de actualizar los pesos. **Learning rate 5e-5:** qué tan grandes son esos ajustes.

In [ ]:
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

set_seed(SEED)
NOMBRE_MODELO = "distilbert-base-multilingual-cased"
tokenizador = AutoTokenizer.from_pretrained(NOMBRE_MODELO)

def a_dataset(d):
    ds = Dataset.from_pandas(d[["titular", "y"]].rename(columns={"y": "label"}), preserve_index=False)
    return ds.map(lambda lote: tokenizador(lote["titular"], truncation=True, max_length=64), batched=True, remove_columns=["titular"])

ds_train, ds_val, ds_test = a_dataset(train), a_dataset(val), a_dataset(test)

modelo = AutoModelForSequenceClassification.from_pretrained(
    NOMBRE_MODELO, num_labels=len(ETIQUETAS), id2label=ID2LABEL, label2id=LABEL2ID)

def calcular_metricas(p):
    pred = np.argmax(p.predictions, axis=1)
    return metricas(p.label_ids, pred)

opciones = dict(output_dir="modelo/checkpoints", num_train_epochs=3, per_device_train_batch_size=16,
                per_device_eval_batch_size=32, learning_rate=5e-5, weight_decay=0.01,
                save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1_macro",
                save_total_limit=1, logging_steps=10, report_to="none", seed=SEED,
                fp16=torch.cuda.is_available())
try:
    args = TrainingArguments(eval_strategy="epoch", **opciones)
except TypeError:  # versiones antiguas de transformers
    args = TrainingArguments(evaluation_strategy="epoch", **opciones)

entrenador = Trainer(model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                     data_collator=DataCollatorWithPadding(tokenizador), compute_metrics=calcular_metricas)

t0 = time.time()
entrenador.train()
tiempo_ft = time.time() - t0
print(f"Entrenamiento: {tiempo_ft:.0f} s")

In [ ]:
# Curvas por época: ¿hay sobreajuste? (pérdida de train baja pero la de validación sube)
log = pd.DataFrame(entrenador.state.log_history)
ev = log.dropna(subset=["eval_f1_macro"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]]
ev

## 6. Evaluación final en test

Este es el único momento en que se usa el conjunto de prueba. Reporta **accuracy**, **F1 macro**, la **matriz de confusión** (filas = categoría real, columnas = predicha; la diagonal son los aciertos) y el desempeño por categoría.

In [ ]:
pred_out = entrenador.predict(ds_test)
logits = pred_out.predictions
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
test["pred_ft"] = [ID2LABEL[i] for i in logits.argmax(axis=1)]
test["prob_ft"] = probs.max(axis=1)

res_ft = metricas(test["y"], logits.argmax(axis=1))
print(f"Fine-tuning (test, {len(test)} titulares) -> accuracy {res_ft['accuracy']:.3f} | F1 macro {res_ft['f1_macro']:.3f}\n")
print(classification_report(test["etiqueta"], test["pred_ft"], labels=ETIQUETAS, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(test["etiqueta"], test["pred_ft"], labels=ETIQUETAS)
plt.figure(figsize=(6.5, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=ETIQUETAS, yticklabels=ETIQUETAS)
plt.xlabel("Categoría predicha"); plt.ylabel("Categoría real")
plt.title(f"Matriz de confusión en test (F1 macro = {res_ft['f1_macro']:.2f})")
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig("modelo/matriz_confusion.png", dpi=150)
plt.show()

In [ ]:
# Ayuda para la reflexión: F1 por categoría, confusiones más frecuentes y errores concretos.
f1_cat = pd.Series(f1_score(test["etiqueta"], test["pred_ft"], labels=ETIQUETAS, average=None), index=ETIQUETAS).round(3)
print("F1 por categoría:\n", f1_cat.sort_values(), "\n")
errores = test[test["etiqueta"] != test["pred_ft"]]
print("Confusiones más frecuentes (real -> predicha):")
print(errores.groupby(["etiqueta", "pred_ft"]).size().sort_values(ascending=False).head(5).to_string(), "\n")
errores[["titular", "etiqueta", "pred_ft", "prob_ft"]].head(10)

**Reflexión por categoría (escríbela con TUS números de arriba, 5–8 líneas):**
- ¿Qué categoría tuvo mejor F1 y por qué crees que es más fácil (vocabulario propio)?
- ¿Cuál fue la peor y con cuál se confundió? Cita 1–2 titulares concretos mal clasificados.
- ¿El problema es del modelo, de los datos (pocos ejemplos, etiquetas por sección que a veces no reflejan el tema) o de la ambigüedad real del titular?
- ¿Qué harías para mejorar: más datos, otro modelo (BETO), más épocas, limpiar etiquetas?

## 7. Comparación y justificación

La tabla compara los tres enfoques. Zero-shot y fine-tuning se miden también sobre **los mismos 40 titulares** para que la comparación sea justa.

In [ ]:
idx40 = test["titular"].isin(muestra40["titular"])
res_ft_40 = metricas(test.loc[idx40, "y"], test.loc[idx40, "pred_ft"].map(LABEL2ID))

tabla = pd.DataFrame({
    "Enfoque": ["Zero-shot (xlm-roberta-large-xnli)", "Fine-tuning (distilbert multilingüe)", "LLM (Gemini)"],
    "Titulares evaluados": [40, f"{len(test)} (y 40 comparables)", len(en_test)],
    "Accuracy": [res_zs["accuracy"], res_ft["accuracy"], res_llm_test["accuracy"]],
    "F1 macro": [res_zs["f1_macro"], res_ft["f1_macro"], res_llm_test["f1_macro"]],
    "F1 macro en los 40": [res_zs["f1_macro"], res_ft_40["f1_macro"], np.nan],
    "Tiempo": [f"{tiempo_zs:.0f} s (inferencia)", f"{tiempo_ft:.0f} s (entrenar)", "~1–2 min (API)"],
    "Costo": ["gratis", "gratis (GPU Colab)", "gratis (cuota) / pago por token"],
}).round(3)
tabla

**Justificación (escríbela tú, 6–8 líneas).** Responde con los números de la tabla:
- ¿Cuál enfoque elegirías para producción y por qué? Considera: calidad (F1 macro), velocidad al predecir, costo, y necesidad de datos etiquetados.
- ¿Cuánto ganó el fine-tuning respecto al zero-shot? ¿Justifica el esfuerzo de etiquetar y entrenar?
- ¿Qué ventaja tiene el LLM (no requiere entrenar) y qué riesgos (costo por uso, dependencia de una API, falta de control)?
- Limitaciones de tu estudio: una sola fuente (El Comercio), ~385 titulares, 4 categorías, etiquetas basadas en la sección del medio.

## 8. Guardar resultados (archivos livianos)

Los **pesos del modelo pesan ~500 MB**: GitHub rechaza archivos de más de 100 MB, así que **no se suben al repositorio** (`.gitignore` ya bloquea `*.safetensors` y `*.bin`). Subimos solo métricas, mapa de etiquetas, imagen de la matriz y predicciones; los pesos se publican en Hugging Face Hub (gratis) para que el dashboard de la Parte 3 los descargue.

In [ ]:
metricas_finales = {
    "f1_macro_test": round(res_ft["f1_macro"], 4), "accuracy_test": round(res_ft["accuracy"], 4),
    "zero_shot_accuracy_40": round(res_zs["accuracy"], 4), "zero_shot_f1_macro_40": round(res_zs["f1_macro"], 4),
    "acuerdo_llm_humano_de_20": acuerdo_20, "acuerdo_llm_vs_seccion": round(float(acuerdo_total), 4),
    "modelo_base": NOMBRE_MODELO, "modelo_llm": MODELO_LLM, "n_train": len(train), "n_val": len(val), "n_test": len(test),
}
json.dump(metricas_finales, open("modelo/metricas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump({"id2label": ID2LABEL, "label2id": LABEL2ID}, open("modelo/etiquetas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
test[["titular", "medio", "etiqueta", "pred_ft", "prob_ft"]].to_csv("datos/predicciones_test.csv", index=False)
print(json.dumps(metricas_finales, ensure_ascii=False, indent=2))

In [ ]:
# OPCIONAL: publicar el modelo en Hugging Face Hub (necesita el secreto HF_TOKEN con permiso de escritura).
SUBIR_A_HF = False
REPO_HF = "gabriela-pozo/noticias-pe-distilbert"   # cambia el usuario si hace falta

if SUBIR_A_HF:
    from huggingface_hub import login
    login(leer_secreto("HF_TOKEN"))
    entrenador.model.push_to_hub(REPO_HF)
    tokenizador.push_to_hub(REPO_HF)
    print("Modelo publicado en https://huggingface.co/" + REPO_HF)

In [ ]:
# Línea para el comentario de entrega en el Issue #188:
n = metricas_finales
print(f"Resultados Test: F1 Macro {n['f1_macro_test']:.2f} | Zero-shot {n['zero_shot_accuracy_40']:.2f} (accuracy en 40) | "
      f"Acuerdo LLM-Humano {n['acuerdo_llm_humano_de_20']}/20")

In [ ]:
# Descarga los archivos livianos para subirlos al repositorio (solo funciona en Colab).
try:
    from google.colab import files
    for ruta in ["datos/etiquetas_llm.csv", "datos/split.csv", "datos/predicciones_test.csv",
                 "modelo/metricas.json", "modelo/etiquetas.json", "modelo/matriz_confusion.png"]:
        files.download(ruta)
except ImportError:
    print("No estás en Colab: los archivos ya están en tu carpeta.")